# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv(r'C:\Users\Sakshit\Desktop\clg projects\SEM 2\Data Visualization\data-viz-class-material\data\co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [6]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: <StringArray>
[ 'United States',          'China',          'India',        'Germany',
 'United Kingdom',         'France',         'Brazil',          'Japan',
         'Canada',      'Australia',    'South Korea',         'Russia',
   'South Africa',         'Mexico',      'Indonesia']
Length: 15, dtype: str

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [11]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE
import plotly.graph_objects as go

# Filter for Asia
asia_df = df[df['region'] == 'Asia']

# Choose the country to highlight
highlight = "India"   # change this if you want

fig = go.Figure()

# 1. Add all Asian countries in grey
for country in asia_df['country'].unique():
    country_data = asia_df[asia_df['country'] == country]
    
    fig.add_trace(go.Scatter(
        x=country_data['year'],
        y=country_data['co2_mt'],   # <-- FIXED COLUMN NAME
        mode='lines',
        line=dict(color='#DDDDDD', width=1),
        showlegend=False
    ))

# 2. Add highlighted country in red
highlight_data = asia_df[asia_df['country'] == highlight]

fig.add_trace(go.Scatter(
    x=highlight_data['year'],
    y=highlight_data['co2_mt'],   # <-- FIXED COLUMN NAME
    mode='lines+text',
    line=dict(color='red', width=3),
    text=[highlight] + [""] * (len(highlight_data) - 1),
    textposition='middle right',
    textfont=dict(color='red', size=14),
    showlegend=False
))

# 3. Layout
fig.update_layout(
    title=f"CO₂ Emissions Over Time — Highlight: {highlight}",
    xaxis_title="Year",
    yaxis_title="CO₂ Emissions (Mt)",
    template="plotly_white",
    height=500
)

fig.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [12]:
# Task 2 — Slopegraph: regional averages
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

# 1. Compute regional averages for 2000 and 2022
regional_avg = (
    df.groupby(['region', 'year'])['co2_mt']
      .mean()
      .reset_index()
)

# Filter only 2000 and 2022
slope = regional_avg[regional_avg['year'].isin([2000, 2022])]

# Pivot for easier comparison
pivot = slope.pivot(index='region', columns='year', values='co2_mt').reset_index()
pivot.columns = ['region', 'y2000', 'y2022']

# Determine color: increase = red, decrease = green
pivot['color'] = pivot.apply(lambda row: 'red' if row['y2022'] > row['y2000'] else 'green', axis=1)

fig = go.Figure()

# 2. Add one line per region
for _, row in pivot.iterrows():
    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[row['y2000'], row['y2022']],
        mode='lines+markers+text',
        line=dict(color=row['color'], width=3),
        text=[f"{row['region']} {row['y2000']:.1f}", f"{row['y2022']:.1f}"],
        textposition=['middle left', 'middle right'],
        showlegend=False
    ))

# 3. Layout formatting
fig.update_layout(
    title="Regional CO₂ Emissions Change (2000 → 2022)",
    xaxis=dict(title="", tickmode='array', tickvals=[2000, 2022]),
    yaxis=dict(title="", showticklabels=False),
    template="plotly_white",
    height=550
)

fig.show()
